# Multi-Person Detection & Re-ID — Run on Google Colab

Runs the full pipeline (YOLOv8 detection → DeepSORT tracking → Re-ID) behind the same FastAPI web console you use locally, exposed through a public HTTPS URL.

**Setup:**
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save
2. Run the cells below **top to bottom**
3. The last cell prints a `https://xxxx.trycloudflare.com` URL — open it in your browser

> Note: your GitHub `main` is behind your local copy, so this notebook uses a project **zip** built from your PC (code + YOLO weights + registered identity DB).

In [ ]:
# ── Cell 1: Upload the project zip ─────────────────────────────────────────
import os, zipfile, shutil
from google.colab import files

os.chdir("/content")          # uploads land here regardless of earlier %cd
DEST = "/content/project"
print("Select D:\\multi-person-detection-reidentification_colab.zip")
up = files.upload()
zip_path = "/content/" + next(iter(up))

if os.path.exists(DEST):
    shutil.rmtree(DEST)
os.makedirs(DEST)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(DEST)
print("Extracted", len(os.listdir(DEST)), "entries to", DEST)
print(sorted(os.listdir(DEST)))

In [ ]:
# ── Cell 2: Install dependencies (PyTorch is already on Colab) ────────────
INSTALL_FACE_RECOGNITION = False  # True = ~10 min dlib build; InsightFace already covers face matching

!pip install -q ultralytics deep-sort-realtime "fastapi>=0.110" "uvicorn>=0.29" python-multipart torchreid==0.2.5 gdown insightface onnxruntime PyYAML tqdm scikit-learn matplotlib scipy

if INSTALL_FACE_RECOGNITION:
    !apt -qq install -y cmake > /dev/null
    !pip install -q face_recognition

In [ ]:
# ── Cell 3: Sanity check ──────────────────────────────────────────────────
%cd /content/project
import sys, os, torch
sys.path.insert(0, "/content/project")

assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU, then re-run all cells"
print("GPU:", torch.cuda.get_device_name(0))

import cv2, ultralytics, fastapi, uvicorn, torchreid, insightface, onnxruntime  # noqa
assert os.path.exists("models/yolov8s.pt"), "models/yolov8s.pt missing — re-upload the zip"
assert os.path.exists("outputs/registration/identity_db.json"), "identity DB missing"
print("All imports OK — ready to launch")

In [ ]:
# ── Cell 4: Start web server + expose public URL ──────────────────────────
import os, sys, time, socket, subprocess, re

os.chdir("/content/project")

# 1) FastAPI server — same entrypoint you run locally (python web/server.py)
log = open("/content/server.log", "w")
server = subprocess.Popen([sys.executable, "web/server.py"], stdout=log, stderr=subprocess.STDOUT)

for _ in range(90):
    s = socket.socket()
    up = s.connect_ex(("127.0.0.1", 8000)) == 0
    s.close()
    if up:
        print("Server is up on port 8000")
        break
    if server.poll() is not None:
        print(open("/content/server.log").read())
        raise RuntimeError("Server process exited — see log above")
    time.sleep(1)
else:
    raise RuntimeError("Server did not open port 8000 in time")

# 2) Public HTTPS URL via Cloudflare quick tunnel (no account needed)
if not os.path.exists("cloudflared"):
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "cloudflared",
    ], check=True)
    os.chmod("cloudflared", 0o755)

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

url = None
t0 = time.time()
while time.time() - t0 < 60:
    line = tunnel.stdout.readline()
    if line:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break
    else:
        time.sleep(0.25)

if url:
    print()
    print("=" * 62)
    print("OPEN THE WEB CONSOLE HERE:", url)
    print("=" * 62)
else:
    print("Tunnel did not report a URL — just re-run this cell.")

## Usage notes
- **First job only**: OSNet (torchreid) and InsightFace `buffalo_s` weights download automatically (~1 min extra).
- Workflow is identical to local: **Register People** → **New Session** (upload camera videos) → watch progress / results / alerts.
- The public URL changes every time you re-run Cell 4.
- Keep the Colab tab open; free-tier runtimes disconnect when idle.
- If the runtime restarts (idle timeout / reconnect): re-run ALL cells from Cell 1 - pip installs are wiped. Original note: re-run Cells 3 → 4 (skip 1–2 unless the disk was wiped).
- To restart the server after a code change: `!pkill -f web/server.py` then re-run Cell 4.